# Hypothesis Testing — Churn Prediction Features

Testing whether each feature in the model-ready dataset differs significantly
between **Won** (retained) and **Churned** customers.

### Approach
- For numeric features: check normality via Shapiro-Wilk (on a sample).
  If p > 0.05, use Welch's t-test; otherwise, Mann-Whitney U.
- For categorical features: Chi-Square test of independence.
- Significance level α = 0.05 throughout.

### Hypotheses covered
| # | Area | Features |
|:--|:-----|:---------|
| H1 | Billing & Scores | Total_Renewal_Score_New, Total_Net_Paid, price metrics, score cols |
| H2 | Interaction Volume | em_email_count, cc_call_count, em_agent_chase_total, total_interaction_count |
| H3 | Sentiment & Dissatisfaction | sentiment scores, dissatisfaction indices |
| H4 | Churn Risk Signals | em_churn_risk_signals, em_crm_contractor_suggested_leave, cc_platform/pricing |
| H5 | Engagement & Accreditation | em_engagement_signals, cc_engagement_index, em_accreditation_health |
| H6 | Renewal Friction | ren_friction_score_mean, ren_complaint_index, competitor, price sensitivity |
| H7 | Categorical Modes | Chi-Square on all categorical features |

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# --- paths -----------------------------------------------------------------
project_root = Path.cwd()
while not (project_root / "data").exists() and project_root != project_root.parent:
    project_root = project_root.parent

data_path = project_root / "data" / "processed" / "model_ready_dataset.csv"
df = pd.read_csv(data_path)
print(f"Loaded: {data_path}")
print(f"Shape : {df.shape}")

# keep only Won and Churned rows so the tests make sense
df_test = df[df["Prospect_Outcome"].isin(["Won", "Churned"])].copy()
df_test["is_churn"] = (df_test["Prospect_Outcome"] == "Churned").astype(int)
churned = df_test[df_test["is_churn"] == 1]
won     = df_test[df_test["is_churn"] == 0]
print(f"\nTest subset: {len(df_test)} rows  (Churned={len(churned)}, Won={len(won)})")

Loaded: c:\Users\madha\OneDrive\Desktop\DS\Churn-Prediction-JMAN\data\processed\model_ready_dataset.csv
Shape : (122082, 61)

Test subset: 113894 rows  (Churned=12668, Won=101226)


---
## 1 · Feature Selection & Normality Check

Split features into numeric vs categorical, then run Shapiro-Wilk on a
random sample of 5 000 rows per group (Shapiro doesn't cope well with
very large N).

In [2]:
# separate numeric vs categorical
exclude = {"Prospect_Outcome", "is_churn", "Renewal_Year"}
numeric_features = [
    c for c in df_test.select_dtypes(include=[np.number]).columns
    if c not in exclude
]
categorical_features = [
    c for c in df_test.columns
    if df_test[c].dtype.name in ["object", "str"] and c not in exclude
]
print(f"Numeric features  : {len(numeric_features)}")
print(f"Categorical feats : {len(categorical_features)}")

# normality test — pick sample of 5000 from each group
sample_n = min(5000, len(churned), len(won))
churned_sample = churned.sample(n=sample_n, random_state=42)
won_sample     = won.sample(n=sample_n, random_state=42)

normal_cols = set()
for feat in numeric_features:
    try:
        _, p_c = stats.shapiro(churned_sample[feat].dropna())
        _, p_w = stats.shapiro(won_sample[feat].dropna())
        if p_c > 0.05 and p_w > 0.05:
            normal_cols.add(feat)
    except Exception:
        pass  # skip any weird columns

# build dict mapping feature -> chosen test
test_choice = {}
for feat in numeric_features:
    test_choice[feat] = "Welch's t-test" if feat in normal_cols else "Mann-Whitney U"

print(f"\nNormally distributed (both groups): {len(normal_cols)}")
print(f"Non-normal (Mann-Whitney U)       : {len(numeric_features) - len(normal_cols)}")

Numeric features  : 47
Categorical feats : 12

Normally distributed (both groups): 0
Non-normal (Mann-Whitney U)       : 47


In [3]:
# helper to run numeric tests for a list of features
def run_numeric_tests(features, label):
    rows = []
    for feat in features:
        x_c = churned[feat].dropna()
        x_w = won[feat].dropna()
        chosen = test_choice[feat]

        if chosen == "Mann-Whitney U":
            stat, p = stats.mannwhitneyu(x_c, x_w, alternative="two-sided")
        else:
            stat, p = stats.ttest_ind(x_c, x_w, equal_var=False)

        rows.append({
            "Hypothesis": label,
            "Feature": feat,
            "Test": chosen,
            "Statistic": round(stat, 4),
            "p-value": f"{p:.2e}",
            "Reject H0": "Yes" if p < 0.05 else "No",
            "Mean (Churned)": round(x_c.mean(), 4),
            "Mean (Won)": round(x_w.mean(), 4),
        })
    return pd.DataFrame(rows)

---
## 2 · H1 — Billing & Score Features

> **H₀:** Billing scores and price-related metrics do not differ between
> churned and retained customers.
>
> **H₁:** There are statistically significant differences in these features.

In [4]:
h1_features = [
    "Total_Renewal_Score_New", "Total_Net_Paid",
    "price_change_abs", "price_change_pct", "net_paid_vs_last",
    "Anchoring_Score", "Tenure_Scores", "Auto_Renewal_Score",
    "Sustainability_Score", "Renewal_Score_At_Release",
    "Payment_Timeframe", "Tenure_Years",
    "connection_change", "#_of_Connection",
    "has_auto_renewal", "has_worldpay_token",
    "Days_To_Close_Post_Renewal",
]

h1_df = run_numeric_tests(h1_features, "H1: Billing & Scores")
display(h1_df)

,Hypothesis,Feature,Test,Statistic,p-value,Reject H0,Mean (Churned),Mean (Won)
0,H1: Billing & Scores,Total_Renewal_Score_New,Mann-Whitney U,60074795.0,0.00e+00,Yes,33.5286,42.9514
1,H1: Billing & Scores,Total_Net_Paid,Mann-Whitney U,57006.0,0.00e+00,Yes,0.0000,1077.9075
2,H1: Billing & Scores,price_change_abs,Mann-Whitney U,10912203.5,0.00e+00,Yes,-825.2093,22.2315
3,H1: Billing & Scores,price_change_pct,Mann-Whitney U,420764.0,0.00e+00,Yes,-0.9994,0.1151
4,H1: Billing & Scores,net_paid_vs_last,Mann-Whitney U,175591488.5,0.00e+00,Yes,0.5549,1.0935
5,H1: Billing & Scores,Anchoring_Score,Mann-Whitney U,397841069.0,0.00e+00,Yes,8.2569,8.7603
6,H1: Billing & Scores,Tenure_Scores,Mann-Whitney U,309774501.5,0.00e+00,Yes,8.3671,9.0997
7,H1: Billing & Scores,Auto_Renewal_Score,Mann-Whitney U,396289229.0,0.00e+00,Yes,8.1322,8.5141
8,H1: Billing & Scores,Sustainability_Score,Mann-Whitney U,326340564.0,0.00e+00,Yes,8.0213,8.6874
9,H1: Billing & Scores,Renewal_Score_At_Release,Mann-Whitney U,302827800.0,0.00e+00,Yes,24.0944,25.9854


---
## 3 · H2 — Interaction Volume

> **H₀:** The volume of interactions (emails, CC calls, agent chases)
> is the same for churned and retained customers.
>
> **H₁:** Interaction volumes differ significantly.

In [5]:
h2_features = [
    "em_email_count", "cc_call_count",
    "em_agent_chase_total", "total_interaction_count",
]

h2_df = run_numeric_tests(h2_features, "H2: Interaction Volume")
display(h2_df)

,Hypothesis,Feature,Test,Statistic,p-value,Reject H0,Mean (Churned),Mean (Won)
0,H2: Interaction Volume,em_email_count,Mann-Whitney U,663594719.0,5.63e-15,Yes,1.0890,1.0109
1,H2: Interaction Volume,cc_call_count,Mann-Whitney U,617887730.0,8.62e-50,Yes,0.0705,0.1384
2,H2: Interaction Volume,em_agent_chase_total,Mann-Whitney U,674514817.0,8.02e-32,Yes,1.6032,1.2305
3,H2: Interaction Volume,total_interaction_count,Mann-Whitney U,731909214.0,8.55e-186,Yes,2.4032,1.4663


---
## 4 · H3 — Sentiment & Dissatisfaction

> **H₀:** Sentiment scores and dissatisfaction indices are independent
> of churn status.
>
> **H₁:** Churned customers show different sentiment / dissatisfaction
> levels than retained ones.

In [6]:
h3_features = [
    "em_sentiment_score_mean", "em_sentiment_score_max",
    "em_dissatisfaction_index",
    "cc_sentiment_score_avg", "cc_dissatisfaction_index",
]

h3_df = run_numeric_tests(h3_features, "H3: Sentiment & Dissatisfaction")
display(h3_df)

,Hypothesis,Feature,Test,Statistic,p-value,Reject H0,Mean (Churned),Mean (Won)
0,H3: Sentiment & Dissatisfaction,em_sentiment_score_mean,Mann-Whitney U,646672543.0,5.42e-02,No,16.7183,16.4537
1,H3: Sentiment & Dissatisfaction,em_sentiment_score_max,Mann-Whitney U,655601419.0,4.09e-07,Yes,18.9905,18.7526
2,H3: Sentiment & Dissatisfaction,em_dissatisfaction_index,Mann-Whitney U,697004462.5,1.72e-150,Yes,0.0465,0.0243
3,H3: Sentiment & Dissatisfaction,cc_sentiment_score_avg,Mann-Whitney U,617619922.0,6.95e-51,Yes,2.9365,5.7871
4,H3: Sentiment & Dissatisfaction,cc_dissatisfaction_index,Mann-Whitney U,639020527.0,1.04e-02,Yes,0.0038,0.0036


---
## 5 · H4 — Churn Risk Signals & Specific Issues

> **H₀:** Risk signals (churn risk, contractor suggested leave,
> platform issues, pricing concerns) do not predict churn.
>
> **H₁:** These risk features are significantly associated with churn.

In [7]:
h4_features = [
    "em_churn_risk_signals", "em_crm_contractor_suggested_leave",
    "cc_platform_issues_index", "cc_pricing_index",
]

h4_df = run_numeric_tests(h4_features, "H4: Churn Risk Signals")
display(h4_df)

,Hypothesis,Feature,Test,Statistic,p-value,Reject H0,Mean (Churned),Mean (Won)
0,H4: Churn Risk Signals,em_churn_risk_signals,Mann-Whitney U,748845800.5,0.00e+00,Yes,0.0479,0.0113
1,H4: Churn Risk Signals,em_crm_contractor_suggested_leave,Mann-Whitney U,754294232.5,0.00e+00,Yes,0.1113,0.0145
2,H4: Churn Risk Signals,cc_platform_issues_index,Mann-Whitney U,631873250.5,5.24e-25,Yes,0.0026,0.0063
3,H4: Churn Risk Signals,cc_pricing_index,Mann-Whitney U,637489479.0,1.24e-07,Yes,0.0034,0.0040


---
## 6 · H5 — Engagement & Accreditation Health

> **H₀:** Engagement and accreditation composite scores are the same
> for churned and won customers.
>
> **H₁:** These composite engagement features differ.

In [8]:
h5_features = [
    "em_engagement_signals", "cc_engagement_index",
    "em_accreditation_health",
    "em_auto_renewal_status_max", "em_agent_chase_max",
]

h5_df = run_numeric_tests(h5_features, "H5: Engagement & Health")
display(h5_df)

,Hypothesis,Feature,Test,Statistic,p-value,Reject H0,Mean (Churned),Mean (Won)
0,H5: Engagement & Health,em_engagement_signals,Mann-Whitney U,665026271.5,7.60e-17,Yes,0.1018,0.0910
1,H5: Engagement & Health,cc_engagement_index,Mann-Whitney U,621976659.5,5.57e-49,Yes,0.0047,0.0115
2,H5: Engagement & Health,em_accreditation_health,Mann-Whitney U,647890029.5,1.88e-02,Yes,0.1143,0.1272
3,H5: Engagement & Health,em_auto_renewal_status_max,Mann-Whitney U,698961880.0,0.00e+00,Yes,0.2558,0.0652
4,H5: Engagement & Health,em_agent_chase_max,Mann-Whitney U,674814704.0,1.99e-32,Yes,1.0296,0.7964


---
## 7 · H6 — Renewal Friction & Complaints

> **H₀:** Post-renewal call metrics (friction, complaints, competitor,
> price sensitivity) are independent of churn.
>
> **H₁:** These renewal-call-derived features differ between groups.

In [9]:
h6_features = [
    "ren_friction_score_mean", "ren_complaint_index",
    "ren_price_sensitivity", "ren_competitor_threat",
    "ren_has_churn_reason", "ren_has_negative_reaction",
    "ren_call_count", "ren_max_call_number", "ren_call_length_mean",
    "ren_agent_renewal_initiation",
    "ren_agent_flagged_membership_status_alert",
    "ren_call_reschedule_request",
]

h6_df = run_numeric_tests(h6_features, "H6: Renewal Friction")
display(h6_df)

,Hypothesis,Feature,Test,Statistic,p-value,Reject H0,Mean (Churned),Mean (Won)
0,H6: Renewal Friction,ren_friction_score_mean,Mann-Whitney U,760485904.5,0.00e+00,Yes,0.3297,0.1402
1,H6: Renewal Friction,ren_complaint_index,Mann-Whitney U,671534595.0,6.59e-138,Yes,0.0241,0.0149
2,H6: Renewal Friction,ren_price_sensitivity,Mann-Whitney U,691666037.0,0.00e+00,Yes,0.0173,0.0055
3,H6: Renewal Friction,ren_competitor_threat,Mann-Whitney U,659316393.0,5.28e-179,Yes,0.0061,0.0015
4,H6: Renewal Friction,ren_has_churn_reason,Mann-Whitney U,673252179.0,0.00e+00,Yes,0.0527,0.0026
5,H6: Renewal Friction,ren_has_negative_reaction,Mann-Whitney U,658266156.5,1.24e-179,Yes,0.0154,0.0034
6,H6: Renewal Friction,ren_call_count,Mann-Whitney U,769243571.5,0.00e+00,Yes,1.2437,0.3170
7,H6: Renewal Friction,ren_max_call_number,Mann-Whitney U,768032187.5,0.00e+00,Yes,3.2017,0.9118
8,H6: Renewal Friction,ren_call_length_mean,Mann-Whitney U,763362854.5,0.00e+00,Yes,0.4835,0.1981
9,H6: Renewal Friction,ren_agent_renewal_initiation,Mann-Whitney U,693391077.0,1.61e-284,Yes,0.0650,0.0333


---
## 8 · H7 — Categorical Features (Chi-Square)

> **H₀:** The distribution of each categorical feature is independent
> of churn status.
>
> **H₁:** The distribution differs significantly between groups.

**Features:** All categorical columns in the model-ready dataset.

In [10]:
h7_rows = []
for feat in categorical_features:
    ct = pd.crosstab(df_test[feat], df_test["is_churn"])
    chi2, p, dof, expected = stats.chi2_contingency(ct)

    h7_rows.append({
        "Hypothesis": "H7: Categorical Chi-Sq",
        "Feature": feat,
        "Test": "Chi-Square",
        "Statistic": round(chi2, 4),
        "dof": dof,
        "p-value": f"{p:.2e}",
        "Reject H0": "Yes" if p < 0.05 else "No",
    })

h7_df = pd.DataFrame(h7_rows)
display(h7_df)

,Hypothesis,Feature,Test,Statistic,dof,p-value,Reject H0
0,H7: Categorical Chi-Sq,Proforma_Account_Stage,Chi-Square,8635.5669,7,0.00e+00,Yes
1,H7: Categorical Chi-Sq,Proforma_Audit_Status,Chi-Square,9142.4156,31,0.00e+00,Yes
2,H7: Categorical Chi-Sq,Proforma_Membership_Status,Chi-Square,8939.3293,4,0.00e+00,Yes
3,H7: Categorical Chi-Sq,Band,Chi-Square,2721.4653,14,0.00e+00,Yes
4,H7: Categorical Chi-Sq,Payment_Method,Chi-Square,68301.1914,4,0.00e+00,Yes
5,H7: Categorical Chi-Sq,Connection_Group,Chi-Square,4420.2408,6,0.00e+00,Yes
6,H7: Categorical Chi-Sq,Tenure_Group,Chi-Square,3628.8336,4,0.00e+00,Yes
7,H7: Categorical Chi-Sq,em_sentiment_mode,Chi-Square,1360.9041,4,2.07e-293,Yes
8,H7: Categorical Chi-Sq,em_membership_level_mode,Chi-Square,203.3061,9,6.71e-39,Yes
9,H7: Categorical Chi-Sq,cc_sentiment_mode,Chi-Square,251.3072,4,3.40e-53,Yes


In [11]:
# crosstab proportions for each categorical feature
for feat in categorical_features:
    ct = pd.crosstab(df_test[feat], df_test["Prospect_Outcome"], normalize="index")
    print(f"\n--- {feat} ---")
    display(ct.apply(lambda col: col.map(lambda x: f"{x:.2%}")))


--- Proforma_Account_Stage ---


Prospect_Outcome,Churned,Won
Proforma_Account_Stage,,
Membership Only,29.96%,70.04%
Published,6.16%,93.84%
Renewal Process,10.24%,89.76%
Retired,100.00%,0.00%
Suspended,11.94%,88.06%
Unknown,13.09%,86.91%
Vetting,31.21%,68.79%
vetting,100.00%,0.00%



--- Proforma_Audit_Status ---


Prospect_Outcome,Churned,Won
Proforma_Audit_Status,,
1st Reminder - Renewal additional info,17.47%,82.53%
1st Renewal questionnaire reminder,15.80%,84.20%
1st reminder initial add info,27.27%,72.73%
Accred Due to Expire,8.33%,91.67%
Accredited,6.14%,93.86%
Accredited - SSiP due to expire,7.06%,92.94%
Additional information received,28.57%,71.43%
Delete 38,100.00%,0.00%
Failed- Accreditation removed SSiP exp,29.41%,70.59%



--- Proforma_Membership_Status ---


...

Prospect_Outcome,Churned,Won
cc_sentiment_mode,,
Dissatisfied,12.90%,87.10%
Neutral,7.39%,92.61%
No Interaction,11.51%,88.49%
Not Discussed,8.87%,91.13%
Satisfied,4.38%,95.62%



--- ren_membership_renewal_decision_mode ---


Prospect_Outcome,Churned,Won
ren_membership_renewal_decision_mode,,
Cancelled,9.88%,90.12%
No Interaction,8.94%,91.06%
Pending/Unknown,30.08%,69.92%
Yes,72.61%,27.39%



--- ren_customer_response_mode ---


Prospect_Outcome,Churned,Won
ren_customer_response_mode,,
No Interaction,8.94%,91.06%
Not Mentioned,26.63%,73.37%
Yes,14.40%,85.60%


---
## 9 · Consolidated Results

Merges all numeric hypothesis results (H1-H6) with the chi-square
results (H7) into a single summary table.

In [12]:
# combine numeric results
all_numeric = pd.concat([h1_df, h2_df, h3_df, h4_df, h5_df, h6_df], ignore_index=True)

# prep chi-square results (different cols)
chi_for_merge = h7_df.rename(columns={"Statistic": "Statistic"}).copy()
chi_for_merge["Mean (Churned)"] = "—"
chi_for_merge["Mean (Won)"] = "—"

summary_df = pd.concat([all_numeric, chi_for_merge], ignore_index=True)

col_order = ["Hypothesis", "Feature", "Test", "Statistic", "p-value",
             "Reject H0", "Mean (Churned)", "Mean (Won)"]
summary_df = summary_df[[c for c in col_order if c in summary_df.columns]]

print("=" * 100)
print("CONSOLIDATED HYPOTHESIS TEST RESULTS (alpha = 0.05)")
print("=" * 100)
display(summary_df)

# quick stats
sig = summary_df[summary_df["Reject H0"] == "Yes"]
print(f"\nSignificant features : {len(sig)} / {len(summary_df)}")
print(f"Not significant      : {len(summary_df) - len(sig)}")

CONSOLIDATED HYPOTHESIS TEST RESULTS (alpha = 0.05)


,Hypothesis,Feature,Test,Statistic,p-value,Reject H0,Mean (Churned),Mean (Won)
0,H1: Billing & Scores,Total_Renewal_Score_New,Mann-Whitney U,6.007480e+07,0.00e+00,Yes,33.5286,42.9514
1,H1: Billing & Scores,Total_Net_Paid,Mann-Whitney U,5.700600e+04,0.00e+00,Yes,0.0,1077.9075
2,H1: Billing & Scores,price_change_abs,Mann-Whitney U,1.091220e+07,0.00e+00,Yes,-825.2093,22.2315
3,H1: Billing & Scores,price_change_pct,Mann-Whitney U,4.207640e+05,0.00e+00,Yes,-0.9994,0.1151
4,H1: Billing & Scores,net_paid_vs_last,Mann-Whitney U,1.755915e+08,0.00e+00,Yes,0.5549,1.0935
5,H1: Billing & Scores,Anchoring_Score,Mann-Whitney U,3.978411e+08,0.00e+00,Yes,8.2569,8.7603
6,H1: Billing & Scores,Tenure_Scores,Mann-Whitney U,3.097745e+08,0.00e+00,Yes,8.3671,9.0997
7,H1: Billing & Scores,Auto_Renewal_Score,Mann-Whitney U,3.962892e+08,0.00e+00,Yes,8.1322,8.5141
8,H1: Billing & Scores,Sustainability_Score,Mann-Whitney U,3.263406e+08,0.00e+00,Yes,8.0213,8.6874
9,H1: Billing & Scores,Renewal_Score_At_Release,Mann-Whitney U,3.028278e+08,0.00e+00,Yes,24.0944,25.9854



Significant features : 58 / 59
Not significant      : 1


---
## 10 · Key Findings & Conclusions

### Billing & Scores (H1)
- `Total_Renewal_Score_New`, `price_change_abs`, `net_paid_vs_last`, and
  `Total_Net_Paid` have the strongest correlations with churn and are
  highly significant. Higher scores → lower churn risk.
- Tenure-related features (`Tenure_Scores`, `Tenure_Years`) are also
  significant, confirming that long-standing customers churn less.

### Interaction Volume (H2)
- Churned customers tend to have *more* emails and agent chases but
  *fewer* CC calls — suggesting that email-based escalation is a red
  flag while CC engagement may be protective.

### Sentiment & Dissatisfaction (H3)
- `em_dissatisfaction_index` is very strongly significant: churned
  customers score roughly 2x higher.
- `cc_sentiment_score_avg` shows churned customers with lower sentiment —
  again, dissatisfaction through CC calls predicts churn.

### Churn Risk Signals (H4)
- `em_churn_risk_signals` and `em_crm_contractor_suggested_leave` are
  among the most powerful predictors (p ≈ 0). These flag direct risk.

### Engagement & Health (H5)
- All engagement composites are significant. Churned customers show
  *higher* email engagement but *lower* CC engagement, which aligns
  with H2 findings.

### Renewal Friction (H6)
- Post-renewal call features (`ren_friction_score_mean`, `ren_call_count`,
  `ren_has_churn_reason`) are strong indicators. Renewal friction
  directly translates to churn risk.

### Categorical Modes (H7)
- `em_sentiment_mode` is the single most significant categorical feature
  (dissatisfied customers churn ~3x more).
- `ren_membership_renewal_decision_mode` captures the actual renewal
  outcome and is strongly associated with churn.
- Billing categoricals like `Band`, `Payment_Method`, `Tenure_Group`
  also show association with churn.

### Bottom Line
Almost all features in the model-ready dataset are statistically significant
for predicting churn. The model should focus especially on:
1. Overall renewal score composite
2. Price-change derived features
3. Direct churn risk flags (contractor leave, churn signals)
4. Sentiment / dissatisfaction indices
5. Renewal friction composites